# 📘 Notebook 07: A Snake That Learns

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module B: Learning with a Table · Notebook 3 of 3 in this module · (7 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Train a **Q-learning** agent on the Snake environment and read its learning curve against two baselines
- Record **videos** of the agent before and after learning
- Inspect a learned Q-table and express what it contains in words
- Show with measurements why **exploration** is necessary in this game
- Demonstrate how a badly designed **reward** produces behaviour nobody wanted
- Explain why a better **observation** can matter more than a better algorithm, and where tables reach their limit

> **Prerequisites:** Notebook 05 (the Q-learning update) and Notebook 06 (the Snake environment, its 11-bit observation, and the rule-based baseline).
>
> 🔭 **Why this notebook matters later:** this is the first agent of the course that learns a real game from scratch, and its training loop is the skeleton of every deep reinforcement learning program in Modules C and D. The limits we run into here are the reasons for those modules.

> 📦 **Libraries used in this notebook.** **Gymnasium** and **imageio**, as in Notebook 06. A single training run takes a few seconds. The comparisons that train many agents take up to a minute each.

In [ ]:
%pip install -q "gymnasium>=1.3" imageio imageio-ffmpeg

In [ ]:
import base64
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
from IPython.display import HTML, display


def show_video(frames, fps=15, width=300):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. The game

The next cell is the environment of Notebook 06, unchanged, together with the rule-based player. Run it. You do not need to read it again.

As a reminder: the agent sees 11 yes/no features (danger ahead, right and left; its direction; where the food is), chooses between going straight, turning right and turning left, and receives +10 for eating and −10 for dying.

In [ ]:
class SnakeEnv(gym.Env):
    """Snake on a square grid. The agent sees 11 yes/no features, not the whole board."""

    metadata = {"render_modes": ["rgb_array"], "render_fps": 10}
    DIRECTIONS = [(0, -1), (1, 0), (0, 1), (-1, 0)]        # up, right, down, left, as (dx, dy)

    def __init__(self, size=10, render_mode="rgb_array"):
        self.size = size
        self.render_mode = render_mode
        self.observation_space = gym.spaces.MultiBinary(11)
        self.action_space = gym.spaces.Discrete(3)          # 0 straight on, 1 turn right, 2 turn left

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)                            # prepares self.np_random
        middle = self.size // 2
        self.snake = [(middle, middle), (middle - 1, middle), (middle - 2, middle)]     # head first
        self.direction = 1                                  # moving right
        self.score = 0
        self.steps_since_food = 0
        self._place_food()
        return self._observation(), {"score": self.score}

    def _place_food(self):
        free = [(x, y) for x in range(self.size) for y in range(self.size) if (x, y) not in self.snake]
        self.food = free[int(self.np_random.integers(len(free)))]

    def _cell_ahead(self, direction):
        dx, dy = self.DIRECTIONS[direction]
        return (self.snake[0][0] + dx, self.snake[0][1] + dy)

    def _is_deadly(self, cell):
        x, y = cell
        return not (0 <= x < self.size and 0 <= y < self.size) or cell in self.snake

    def _observation(self):
        d, head, food = self.direction, self.snake[0], self.food
        features = [
            self._is_deadly(self._cell_ahead(d)),               # danger straight ahead
            self._is_deadly(self._cell_ahead((d + 1) % 4)),     # danger on my right
            self._is_deadly(self._cell_ahead((d - 1) % 4)),     # danger on my left
            d == 0, d == 1, d == 2, d == 3,                     # moving up, right, down, left
            food[0] < head[0], food[0] > head[0],               # food is to the left, to the right
            food[1] < head[1], food[1] > head[1],               # food is above, below
        ]
        return np.array(features, dtype=np.int8)

    def step(self, action):
        self.direction = (self.direction + (0, 1, -1)[action]) % 4
        new_head = self._cell_ahead(self.direction)
        self.steps_since_food += 1
        if self._is_deadly(new_head):                           # hit a wall or itself
            cause = "body" if new_head in self.snake else "wall"
            return self._observation(), -10.0, True, False, {"score": self.score, "death": cause}
        self.snake.insert(0, new_head)
        reward, terminated = 0.0, False
        if new_head == self.food:
            self.score += 1
            self.steps_since_food = 0
            reward = 10.0
            if len(self.snake) == self.size ** 2:               # the board is full: a perfect game
                terminated = True
            else:
                self._place_food()
        else:
            self.snake.pop()                                    # no food: the tail moves on
        truncated = self.steps_since_food >= 2 * self.size ** 2     # going round in circles
        return self._observation(), reward, terminated, truncated, {"score": self.score}

    def render(self, cell=24):
        image = np.full((self.size * cell, self.size * cell, 3), 25, dtype=np.uint8)

        def paint(position, colour, margin=2):
            x, y = position
            image[y * cell + margin:(y + 1) * cell - margin, x * cell + margin:(x + 1) * cell - margin] = colour

        paint(self.food, (220, 60, 60))
        for part in self.snake[1:]:
            paint(part, (70, 170, 90))
        paint(self.snake[0], (160, 235, 130), margin=1)
        return image


def rule_based(observation):
    """Go towards the food if that is safe. Otherwise go anywhere that is safe."""
    danger = observation[0:3]                               # for the actions straight, right, left
    heading = int(np.argmax(observation[3:7]))
    wanted = []                                             # directions that bring the head closer to the food
    if observation[7]:
        wanted.append(3)                                    # food to the left: moving left helps
    if observation[8]:
        wanted.append(1)
    if observation[9]:
        wanted.append(0)
    if observation[10]:
        wanted.append(2)
    for action, turn in ((0, 0), (1, 1), (2, -1)):
        if (heading + turn) % 4 in wanted and not danger[action]:
            return action
    for action in (0, 1, 2):
        if not danger[action]:
            return action
    return 0

## 2. From an observation to a row of the table

A Q-table needs a row number for every situation. The observation is a list of 11 bits, and 11 bits are a number between 0 and 2047 written in binary. So the table has 2048 rows, one per possible observation, and 3 columns, one per action.

In [ ]:
POWERS = 2 ** np.arange(11)


def row(observation):
    """Turn the 11 bits into a row number of the Q-table."""
    return int(observation @ POWERS)


env = SnakeEnv()
observation, info = env.reset(seed=0)
print("observation:", observation, "-> row", row(observation))

The helpers below play and measure a policy, as in Notebook 06. A policy here is any function from an observation to an action. `greedy(Q)` builds one from a Q-table.

In [ ]:
def play(env, policy, seed=None, record=False):
    """One game. Returns the score, the number of steps, how the game ended, and the frames."""
    observation, info = env.reset(seed=seed)
    frames, steps = [], 0
    terminated = truncated = False
    while not (terminated or truncated):
        if record:
            frames.append(env.render())
        observation, reward, terminated, truncated, info = env.step(policy(observation))
        steps += 1
    if record:
        frames.append(env.render())
    ending = info.get("death", "time limit" if truncated else "board full")
    return info["score"], steps, ending, frames


def measure(env, policy, games=200):
    """Average score, best score, and how the games ended."""
    scores, endings = [], {}
    for game in range(games):
        score, steps, ending, frames = play(env, policy, seed=50000 + game)
        scores.append(score)
        endings[ending] = endings.get(ending, 0) + 1
    return float(np.mean(scores)), int(np.max(scores)), endings


def greedy(Q):
    return lambda observation: int(Q[row(observation)].argmax())


rng = np.random.default_rng(0)
random_score = measure(env, lambda observation: int(rng.integers(3)))[0]
rules_score = measure(env, rule_based)[0]
print(f"random snake:     average score {random_score:.2f}")
print(f"rule-based snake: average score {rules_score:.1f}")

These are the two numbers to beat. The second one is the interesting question of this notebook: **can an agent that is told nothing match the rules we wrote by hand?**

## 3. The training loop

### Formal definition
The agent plays game after game. After every single step it applies the Q-learning update of Notebook 05:

$$Q(S, A) \leftarrow Q(S, A) + \alpha\,\big[\,R + \gamma\,\max_a Q(S', a) - Q(S, A)\,\big]$$

Exploration is epsilon-greedy. Epsilon starts at 1 (every move random), falls in a straight line during the first 300 games, and stays at 0.01 afterwards.

In [ ]:
def q_learning(env, games, alpha=0.1, gamma=0.9, explore_games=300, epsilon_end=0.01, n_rows=2048, seed=0):
    """Tabular Q-learning. Returns the table and the score of every training game."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((n_rows, 3))
    scores = []
    for game in range(games):
        epsilon = max(epsilon_end, 1 - game / explore_games) if explore_games else epsilon_end
        observation, info = env.reset(seed=seed * 100000 + game)
        state = row(observation)
        terminated = truncated = False
        while not (terminated or truncated):
            if rng.random() < epsilon:
                action = int(rng.integers(3))                                     # explore
            else:
                action = int(rng.choice(np.flatnonzero(Q[state] == Q[state].max())))   # exploit, ties at random
            observation, reward, terminated, truncated, info = env.step(action)
            next_state = row(observation)
            target = reward + gamma * (0.0 if terminated else Q[next_state].max())
            Q[state, action] += alpha * (target - Q[state, action])
            state = next_state
        scores.append(info["score"])
    return Q, np.array(scores)

## 4. Before and after

An untrained table is full of zeros, so every action looks equally good in every situation. Here is how that plays.

In [ ]:
empty_table = np.zeros((2048, 3))
tie_breaker = np.random.default_rng(1)
untrained = lambda observation: int(tie_breaker.integers(3))      # all values equal: the choice is arbitrary

score, steps, ending, frames = play(env, untrained, seed=3, record=True)
print(f"before learning: score {score}, {steps} steps, ended by: {ending}")
show_video(frames)

Now we train for 1,000 games. On an ordinary processor this takes a few seconds.

In [ ]:
Q, training_scores = q_learning(env, games=1000)


def smooth(values, window=25):
    return np.convolve(values, np.ones(window) / window, mode="valid")


plt.figure(figsize=(7, 4))
plt.plot(smooth(training_scores), label="Q-learning (smoothed)")
plt.axhline(rules_score, color="green", linestyle="--", label="hand-written rules")
plt.axhline(random_score, color="red", linestyle="--", label="random")
plt.axvline(300, color="gray", linestyle=":", label="exploration ends")
plt.title("Snake: score per game during training")
plt.xlabel("game"); plt.ylabel("score"); plt.grid(True); plt.legend()
plt.show()

learned_score, best, endings = measure(env, greedy(Q))
print(f"learned policy, no exploration: average score {learned_score:.1f}, best game {best}")

Read the curve from left to right. For the first couple of hundred games nothing seems to happen: the snake is mostly moving at random and scores close to zero. But the table is filling up during that time. As epsilon falls, the agent begins to use what it has collected, and the score climbs steeply.

Evaluated without exploration, the agent reaches about the same average score as our hand-written rules. It was never told that food is good to approach or that walls should be avoided. It was given +10 and −10.

In [ ]:
score, steps, ending, frames = play(env, greedy(Q), seed=3, record=True)
print(f"after learning: score {score}, {steps} steps, ended by: {ending}")
show_video(frames)

## 5. What is in the table?

A neural network is hard to inspect. A Q-table is not: we can read every row. Here are a few situations with the values the agent has learned for its three actions.

In [ ]:
FEATURES = ["danger ahead", "danger right", "danger left",
            "moving up", "moving right", "moving down", "moving left",
            "food left", "food right", "food above", "food below"]
ACTIONS = ["straight", "turn right", "turn left"]


def situation(*names):
    return np.array([name in names for name in FEATURES], dtype=np.int8)


examples = [situation("moving right", "food right"),
            situation("moving right", "food above"),
            situation("moving right", "food below"),
            situation("danger ahead", "moving right", "food above"),
            situation("danger ahead", "moving right", "food left", "food above"),
            situation("danger ahead", "danger right", "moving up", "food left")]

for observation in examples:
    values = Q[row(observation)]
    words = ", ".join(name for name, bit in zip(FEATURES, observation) if bit)
    print(f"{words:52s} straight {values[0]:6.2f}  right {values[1]:6.2f}  left {values[2]:6.2f}   -> {ACTIONS[int(values.argmax())]}")

Go through the rows. Food straight ahead: go straight. A snake moving right with the food above it must turn **left** (its own left is up on the screen), and with the food below it must turn right.

In the rows with danger ahead, going straight has a **negative** value. Every time the agent tried that move, the game ended with −10, and each try moved the value one tenth of the way towards −10 (the step size is 0.1). How negative the entry is tells you how often the agent made the mistake before it stopped. In the last row two of the three moves are deadly, and the agent takes the only one left.

The agent has rebuilt, from rewards, most of the rule we wrote in Notebook 06. How much of it exactly?

In [ ]:
def all_possible_observations():
    """The 256 observations that can really occur (Notebook 06, Exercise 5)."""
    result = []
    for danger in range(8):
        for heading in range(4):
            for food_x in ((0, 0), (1, 0), (0, 1)):
                for food_y in ((0, 0), (1, 0), (0, 1)):
                    if food_x == (0, 0) and food_y == (0, 0):
                        continue
                    bits = [danger & 1, (danger >> 1) & 1, (danger >> 2) & 1]
                    bits += [int(heading == i) for i in range(4)] + list(food_x) + list(food_y)
                    result.append(np.array(bits, dtype=np.int8))
    return result


observations = all_possible_observations()
policy = greedy(Q)
same = sum(policy(o) == rule_based(o) for o in observations)
unsafe = sum(o[policy(o)] == 1 and min(o[0:3]) == 0 for o in observations)

print(f"situations in which the learned policy agrees with the hand-written rule: {same} of {len(observations)}")
print(f"situations in which it steps into visible danger although a safe move exists: {unsafe}")

The two policies agree in three quarters of the situations. Many of the disagreements are harmless: when the food is diagonally ahead there are two sensible moves, and the rule and the agent may simply prefer different ones. The few entries where the learned policy steps into visible danger belong to situations that hardly ever occur, so the agent has had almost no chance to learn about them. A table knows nothing about a row it has not visited.

### The same blind spot

In [ ]:
print("How 200 games ended:")
for name, policy in (("rule-based", rule_based), ("learned", greedy(Q))):
    average, best, endings = measure(env, policy)
    print(f"  {name:11s}", ", ".join(f"{ending}: {count}" for ending, count in sorted(endings.items())))

The learned snake dies as the rule-based snake did: boxed in by its own body. Notebook 06 predicted it. The observation shows only the three cells around the head, so no policy built on it, written or learned, can see a trap forming. The agent has learned about as much as its 11 bits allow. **The algorithm is not what is holding it back. The observation is.**

## 6. Exploration is not optional

In Blackjack (Notebook 04, Exercise 4) an agent that never explored still did tolerably, because the cards put it in a different situation every hand. Snake does not shuffle anything for the agent. Three exploration schedules, five training runs each, 600 games per run:

In [ ]:
schedules = {
    "no exploration":         dict(explore_games=0, epsilon_end=0.0),
    "constant epsilon 0.1":   dict(explore_games=0, epsilon_end=0.1),
    "from 1 down to 0.01":    dict(explore_games=300, epsilon_end=0.01),
}

results = {}
for name, settings in schedules.items():
    results[name] = [measure(env, greedy(q_learning(env, 600, seed=seed, **settings)[0]), games=100)[0]
                     for seed in range(5)]
    print(f"{name:22s} scores of five runs: " + "  ".join(f"{s:5.1f}" for s in results[name])
          + f"   average {np.mean(results[name]):.1f}")

plt.figure(figsize=(7, 4))
for position, (name, scores) in enumerate(results.items()):
    plt.scatter([position] * len(scores), scores, s=40)
plt.xticks(range(len(results)), results.keys())
plt.title("Final score of the greedy policy, five training runs per schedule")
plt.ylabel("average score over 100 games"); plt.grid(True)
plt.show()

Without exploration the result is poor and, just as telling, **unpredictable**: one run stumbles on something reasonable and another learns almost nothing. A snake that has once collected a few points by some habit keeps that habit and never learns what the alternatives are worth.

A constant epsilon of 0.1 is reliable, all five runs end close together, but it ends lower. A likely reason: a snake that makes a random move one step in ten rarely survives long, so it gets little practice in the situations that only a long snake meets, and those rows of the table stay poorly learned.

The decaying schedule is the best of the three: explore widely while nothing is known, then settle down.

## 7. Be careful what you reward

### Intuition first
A company that pays its call centre by the number of calls answered gets many short, useless calls. Agents are the same, only more literal. They maximise the reward **as written**, not the intention behind it.

Suppose someone "improves" our reward with a sensible-sounding idea: staying alive is good, so give +1 for every step survived.

In [ ]:
class SurvivalBonus(SnakeEnv):
    """The same game, with +1 for every step that does not end in death."""

    def step(self, action):
        observation, reward, terminated, truncated, info = super().step(action)
        if not terminated:
            reward += 1.0
        return observation, reward, terminated, truncated, info


Q_bonus, scores = q_learning(SurvivalBonus(), games=1000)

average, best, endings = measure(env, greedy(Q_bonus))          # judged on the real game
print(f"trained with the survival bonus: average score {average:.1f}")
print("how its games ended:", ", ".join(f"{ending}: {count}" for ending, count in sorted(endings.items())))

score, steps, ending, frames = play(env, greedy(Q_bonus), seed=3, record=True)
print(f"this game: score {score}, {steps} steps, ended by: {ending}")
show_video(frames, fps=30)

The score has collapsed, and a large share of the games now end by the time limit. Watch the video: the snake has discovered that turning in circles earns +1 per step for ever at no risk, while chasing food means growing, and growing is dangerous. From the agent's point of view this is not cheating. It is the best answer to the question we asked.

This is called **reward hacking**, and it is one of the practical problems of the whole field. The lesson of Notebook 06, Section 4 stands: reward the **outcome** you want. Notebook 18 returns to this with more examples.

> ⚠️ **Common pitfalls**
>
> - **Judging by the training curve alone.** It includes exploration. Always measure the greedy policy separately.
> - **Evaluating on the reward you invented.** The bonus agent scores very well on *its* reward. Judge every agent on the real objective, here the game score.
> - **Trusting one training run.** Section 6 showed how far runs with different seeds can lie apart.

## 8. A better observation beats a better algorithm

The agent's ceiling is set by its observation, so let us raise the ceiling. The snake dies in pockets it cannot see. We add three features that look further: for each of the three moves, **would it take me into a region with less free space than my own length?** The free space is counted by spreading out from the cell in all directions, a technique called flood fill.

In [ ]:
class RoomSnakeEnv(SnakeEnv):
    """Snake with 3 extra features: is the space behind each move too small for my body?"""

    def __init__(self, size=10, render_mode="rgb_array"):
        super().__init__(size, render_mode)
        self.observation_space = gym.spaces.MultiBinary(14)

    def _room(self, start):
        """Number of free cells reachable from `start`, counted up to the snake's length."""
        if self._is_deadly(start):
            return 0
        reached, to_visit = {start}, [start]
        while to_visit and len(reached) <= len(self.snake):
            x, y = to_visit.pop()
            for dx, dy in self.DIRECTIONS:
                cell = (x + dx, y + dy)
                if cell not in reached and not self._is_deadly(cell):
                    reached.add(cell)
                    to_visit.append(cell)
        return len(reached)

    def _observation(self):
        d = self.direction
        cramped = [self._room(self._cell_ahead(direction)) <= len(self.snake)
                   for direction in (d, (d + 1) % 4, (d - 1) % 4)]
        return np.concatenate([super()._observation(), np.array(cramped, dtype=np.int8)])


POWERS = 2 ** np.arange(14)                      # row() now has to handle 14 bits

room_env = RoomSnakeEnv()
Q_room, scores = q_learning(room_env, games=1500, n_rows=2 ** 14)

average, best, endings = measure(room_env, greedy(Q_room))
print(f"11 features: average score {learned_score:.1f}")
print(f"14 features: average score {average:.1f}, best game {best}")

score, steps, ending, frames = play(room_env, greedy(Q_room), seed=3, record=True)
show_video(frames, fps=20)

Three well-chosen bits raise the score clearly, with the same algorithm and the same reward. The snake now turns away from pockets before entering them.

### Where this road ends
We could keep going: add features for the tail, for the distance to the walls, for the shape of the free space. Each would help a little, each needs a programmer who understands Snake, and none of it would transfer to another game. The table also grows: 14 bits already mean 16,384 rows, most of them never visited.

What we would really like is to hand the agent the **whole board** and let it work out for itself which features matter. A table cannot do that: Notebook 06 showed that almost every board is new. We need something that can **generalise**, that can give a sensible answer in a situation it has never seen because it resembles situations it has seen. That is what neural networks do, and it is the subject of Module C.

---
## ✏️ Exercises

The exercises use `SnakeEnv`, `q_learning`, `measure`, `greedy`, `play` and `show_video`. Section 8 changed `POWERS` to 14 bits. The first exercise cell sets it back, so run the exercises in order.

### Exercise 1 (Reading the table)
Set `POWERS` back to 11 bits. Then print the three Q-values of the table `Q` for the situation "danger right, moving down, food left". Which action does the agent choose? Is it the one you would choose? Remember that directions are seen from the snake's head.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
POWERS = 2 ** np.arange(11)

observation = situation("danger right", "moving down", "food left")
for name, value in zip(ACTIONS, Q[row(observation)]):
    print(f"{name:11s} {value:6.2f}")
print("chosen:", ACTIONS[int(Q[row(observation)].argmax())])
```

*For a snake moving down the screen, its own right is the left of the screen. So the food lies in the direction that is blocked. Turning right is deadly, and the agent must choose between going straight and turning away from the food. Both are safe, and the table tells you which one it has found more profitable.*
</details>

### Exercise 2 (A hint in the reward)
A gentler kind of help than the survival bonus: keep +10 and −10, and on every other step give +1 if the head moved closer to the food and −1 if it moved away. Write this as a subclass `CloserBonus`, train 600 games with three different seeds, and compare with the plain reward on (a) the average training score of the first 150 games and (b) the final score of the greedy policy on the real game. Does the hint help, and does it do harm?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class CloserBonus(SnakeEnv):
    def step(self, action):
        head, food = self.snake[0], self.food
        before = abs(head[0] - food[0]) + abs(head[1] - food[1])
        observation, reward, terminated, truncated, info = super().step(action)
        if reward == 0 and not terminated:
            head = self.snake[0]
            after = abs(head[0] - food[0]) + abs(head[1] - food[1])
            reward = 1.0 if after < before else -1.0
        return observation, reward, terminated, truncated, info


for name, environment in (("plain reward", SnakeEnv()), ("closer bonus", CloserBonus())):
    early, final = [], []
    for seed in range(3):
        table, scores = q_learning(environment, 600, seed=seed)
        early.append(scores[:150].mean())
        final.append(measure(env, greedy(table), games=100)[0])
    print(f"{name:13s} first 150 games: {np.mean(early):.2f}   final greedy score: "
          + "  ".join(f"{s:.1f}" for s in final))
```

*The hint gives the agent feedback on every step, not only on the rare step that reaches the food, so the early games are a little better and the three runs end closer together. The final score is about the same. This hint is fairly safe because it cannot be collected for ever: the gain from moving closer is paid back when moving away. The survival bonus could be collected without limit, and that made it dangerous.*
</details>

### Exercise 3 (Is the penalty needed?)
Train an agent in a version of the game where dying gives a reward of 0 and not −10 (subclass `SnakeEnv` and change the reward in `step`). Use three seeds and 600 games, and compare the final scores with the plain reward. Why does the agent still learn to avoid walls at all?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class NoPenalty(SnakeEnv):
    def step(self, action):
        observation, reward, terminated, truncated, info = super().step(action)
        if terminated and reward < 0:
            reward = 0.0
        return observation, reward, terminated, truncated, info


for name, environment in (("with -10 for dying", SnakeEnv()), ("no penalty", NoPenalty())):
    final = [measure(env, greedy(q_learning(environment, 600, seed=seed)[0]), games=100)[0] for seed in range(3)]
    print(f"{name:20s} final greedy scores: " + "  ".join(f"{s:.1f}" for s in final))
```

*Dying ends the game, and with it every future +10. A move that kills therefore has a value of 0, while a move that keeps the snake alive has a positive value from the food still to come. That difference is enough to teach avoidance, so the penalty is not strictly necessary. It helps, because it makes the difference larger and visible from the first death, long before the agent has learned to find food.*
</details>

### Exercise 4 (Looking ahead, or not)
Train with gamma = 0, 0.5, 0.9 and 0.99 (600 games, three seeds each, plain reward) and print the final greedy scores. Which value is best? The answer may surprise you: try to explain it using what the 11 bits can and cannot tell the agent about the future.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for gamma in (0.0, 0.5, 0.9, 0.99):
    final = [measure(env, greedy(q_learning(env, 600, gamma=gamma, seed=seed)[0]), games=100)[0] for seed in range(3)]
    print(f"gamma = {gamma:<5} final greedy scores: " + "  ".join(f"{s:5.1f}" for s in final))
```

*In most problems a far-sighted agent does better. Here the short-sighted ones do at least as well, and gamma = 0.99 is clearly worse. With gamma = 0 the value of an action is its average **immediate** reward. That is enough to prefer moving towards the food (sometimes the food is in the very next cell, so that direction pays off on average) and to avoid deadly moves. The 11 bits carry almost no information about the more distant future, so looking further ahead cannot add much. What it does add is noise: the same observation is followed by very different futures, and with a large gamma all that variation is passed back through the table.*
</details>

### Exercise 5 (Does the table transfer? (a little harder))
The agent was trained on a 10 × 10 board. Without any further training, let the same table `Q` play on boards of size 6, 14 and 20, and compare with the rule-based snake on each. Then explain why a table indexed by the full board (Notebook 06, Exercise 6) could not be moved to another board size at all.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for size in (6, 10, 14, 20):
    board = SnakeEnv(size=size)
    learned = measure(board, greedy(Q), games=100)[0]
    rules = measure(board, rule_based, games=100)[0]
    print(f"{size:>2} x {size:<2}: learned on 10 x 10: {learned:5.1f}    rules: {rules:5.1f}")
```

*The table works on every board size, because its features (danger next to the head, direction of the food) mean the same thing on any board. A good representation makes knowledge **transferable**. A table with one row per board position has nothing to offer on a board it was not built for: none of its rows exist there.*
</details>

## 🔑 Key takeaways

- Tabular **Q-learning**, given only +10 and −10, learned to play Snake about as well as rules written by hand.
- A Q-table can be **read**: each row is a situation, each entry the learned value of an action in it.
- In this game **exploration** decides between a good and a poor result. Explore widely first, then reduce.
- Agents maximise the reward **as written**. A survival bonus produced a snake that circles for ever: **reward hacking**.
- The agent's ceiling was set by its **observation**. Three extra features lifted it more than any amount of extra training could have.
- Hand-made features do not scale and do not transfer between games. We need agents that **generalise** from raw input.

### 🏁 You have completed Module B!

You have met the two classic model-free algorithms, SARSA and Q-learning, built a game as a Gymnasium environment, and trained an agent on it from nothing. You have also seen where tables stop: when the state is too large to list, and when nobody is there to design the features.

**Module C** replaces the table with a neural network. The update rule will look familiar. Making it work will take two new ideas.

---
*End of Module B.*

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*